# Day 17: The ReAct Pattern

Walkthrough of Reasoning + Acting: writing a thought before every action, reading the observation before deciding what's next.

## 1. What is ReAct

Thought, Action, Action Input, Observation, repeat, until a Final Answer -- the agent loop from Day 15, made explicit as text.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('../day-15-agentic-ai'))
sys.path.insert(0, os.path.abspath('../day-16-custom-tools'))
print("Thought -> Action -> Action Input -> Observation -> repeat -> Final Answer")
print("A visible thought can be read, debugged, and trusted -- a silent decision can't.")

## 2. Parse ReAct text

Turning Thought/Action/Action Input/Final Answer text into a structured object, with clear errors for malformed turns.

In [ ]:
from react_parser import parse_react_text, ReActParseError

step = parse_react_text("Thought: I need to look up the order first.\nAction: lookup_order\nAction Input: ORD-1001")
print(step)

try:
    parse_react_text("Thought: I should look something up.\nAction: lookup_order")
except ReActParseError as exc:
    print(f"Correctly rejected: {exc}")

## 3. Run the ReAct loop

The full loop against Day 16's support tools, printing each Thought/Action/Observation as it runs.

In [ ]:
from tool_registry import ToolRegistry
from tool_builder import build_tool_from_function
from support_tools import lookup_order, check_refund_eligibility
from react_loop import ReActAgent

registry = ToolRegistry()
for fn in (lookup_order, check_refund_eligibility):
    build_tool_from_function(registry, fn)

agent = ReActAgent(registry)
answer = agent.run("Is order ORD-1003 eligible for a refund?")
print(f"Final answer: {answer}")

## 4. ReAct vs plain agent

Same task, same tools: a plain agent that never questions its first assumption, versus a ReAct agent that catches a surprising observation.

In [ ]:
from react_vs_plain_agent import run_plain_agent, run_react_agent

plain_registry = ToolRegistry()
for fn in (lookup_order, check_refund_eligibility):
    build_tool_from_function(plain_registry, fn)

plain_result = run_plain_agent(plain_registry)

react_registry = ToolRegistry()
for fn in (lookup_order, check_refund_eligibility):
    build_tool_from_function(react_registry, fn)

react_answer = run_react_agent(react_registry)
print(f"\nPlain agent result: {plain_result}")
print(f"ReAct agent answer: {react_answer}")

## 5. Thought quality

Weak, good, and bad thoughts -- plus a checker that flags a thought with no vocabulary overlap with the last observation.

In [ ]:
from thought_quality import (
    weak_thought_restates_question, good_thought_plans_next_step,
    bad_thought_ignores_observation, good_thought_checks_assumption,
    thought_ignores_last_observation,
)

last_observation = {"status": "shipped", "order_id": "ORD-1002"}
bad = bad_thought_ignores_observation(last_observation)
good = good_thought_checks_assumption(last_observation)

print(f"Bad thought:  '{bad}' -> flagged: {thought_ignores_last_observation(bad, last_observation)}")
print(f"Good thought: '{good}' -> flagged: {thought_ignores_last_observation(good, last_observation)}")

## 6. Loop guards

Max steps, repeated identical actions, and unknown tools -- each caught with a clear message instead of hanging or crashing.

In [ ]:
from loop_guards import run_guarded_step, LoopGuardTriggered

available_tools = {"lookup_order", "check_refund_eligibility"}
history = [{"action": "lookup_order", "action_input": "ORD-1001", "observation": {}}]

for label, kwargs in [
    ("repeated action", dict(step_number=2, max_steps=10, history=history, action="lookup_order", action_input="ORD-1001", available_tools=available_tools)),
    ("unknown tool", dict(step_number=1, max_steps=10, history=[], action="delete_everything", action_input="all", available_tools=available_tools)),
]:
    try:
        run_guarded_step(**kwargs)
        print(f"{label}: no guard triggered")
    except LoopGuardTriggered as exc:
        print(f"{label}: {exc}")

## 7. ReAct with RAG

The real Day 5 retriever and Day 6 ranker wrapped as a single search tool -- retrieval becomes just another Observation.

In [ ]:
sys.path.insert(0, os.path.abspath('../day-05-retrieval'))
sys.path.insert(0, os.path.abspath('../day-06-ranking'))

from react_with_rag import RagSearchTool, run_react_with_rag

rag_tool = RagSearchTool()
rag_registry = ToolRegistry()
build_tool_from_function(rag_registry, rag_tool.search)

answer = run_react_with_rag(rag_registry, "What is RAG?")
print(f"Final answer: {answer}")

## 8. Key takeaways

- ReAct makes an agent's reasoning explicit text: Thought, Action, Action Input, Observation, repeated until a Final Answer.
- A parser that rejects malformed turns (an Action with no Action Input, neither an Action nor a Final Answer) catches format errors before they become silent bugs.
- The same task run with and without visible thoughts showed a concrete difference: the ReAct agent caught a bad assumption (checking refund eligibility before confirming delivery) that the plain agent never noticed.
- Not every thought is useful -- a thought that merely restates the question or ignores the last observation adds a reasoning step without adding any actual reasoning.
- Loops need guards: max steps, repeated-action detection, and unknown-tool detection all turn a stuck or confused loop into a clear, immediate stop.
- Retrieval plugs into ReAct exactly like any other tool: once wrapped, the Day 5/6 pipeline's output is just another Observation for the agent to read before answering.
- A written thought is only useful if it's true -- the discipline that matters isn't writing SOME thought before every action, it's writing one that genuinely reflects what should happen next.